Extract miRNA seed; 1-based positions 2-7 of miRNA

In [4]:
import pandas as pd
from miRBench.dataset import get_dataset_df 

print("--- AGO2_CLASH_Hejret2023 Dataset ---")
df1 = get_dataset_df('AGO2_CLASH_Hejret2023', split="test")[['gene', 'noncodingRNA', 'label']]
print("\n--- AGO2_eCLIP_Klimentova2022 Dataset ---")
df2 = get_dataset_df('AGO2_eCLIP_Klimentova2022', split="test")[['gene', 'noncodingRNA', 'label']]
datasets = {"AGO2_CLASH_Hejret2023": df1,
    "AGO2_eCLIP_Klimentova2022": df2}

def seed6mer(noncodingRNA):
    return noncodingRNA[1:7]

for name, df in datasets.items():
    df["seed6mer"] = df["noncodingRNA"].apply(seed6mer)
    print(f"\n--- {name} ---")
    display (df['seed6mer'].head())

--- AGO2_CLASH_Hejret2023 Dataset ---
Using cached dataset C:\Users\05vic\.miRBench\datasets\20540907\AGO2_CLASH_Hejret2023\test\dataset.tsv

--- AGO2_eCLIP_Klimentova2022 Dataset ---
Using cached dataset C:\Users\05vic\.miRBench\datasets\20540907\AGO2_eCLIP_Klimentova2022\test\dataset.tsv

--- AGO2_CLASH_Hejret2023 ---


0    AAGCAA
1    ATAGCT
2    GAATTC
3    CTAAGG
4    GAAGCG
Name: seed6mer, dtype: str


--- AGO2_eCLIP_Klimentova2022 ---


0    AAGGTG
1    AAGGTG
2    AAAGTG
3    AAGGTG
4    AAAGTG
Name: seed6mer, dtype: str

Reverse complement (rv) the extracted seed sequence

In [5]:
Comp =  str.maketrans({"A": "T", "T": "A", "C": "G", "G": "C"})

for name, df in datasets.items():
    df['seed6mer_rv'] = df['seed6mer'].str[::-1]
    df["seed6mer_rvc"] = df['seed6mer_rv'].str.translate(Comp)

    print(name)
    display(df[['seed6mer','seed6mer_rv','seed6mer_rvc']].head())

AGO2_CLASH_Hejret2023


,seed6mer,seed6mer_rv,seed6mer_rvc
0,AAGCAA,AACGAA,TTGCTT
1,ATAGCT,TCGATA,AGCTAT
2,GAATTC,CTTAAG,GAATTC
3,CTAAGG,GGAATC,CCTTAG
4,GAAGCG,GCGAAG,CGCTTC


AGO2_eCLIP_Klimentova2022


,seed6mer,seed6mer_rv,seed6mer_rvc
0,AAGGTG,GTGGAA,CACCTT
1,AAGGTG,GTGGAA,CACCTT
2,AAAGTG,GTGAAA,CACTTT
3,AAGGTG,GTGGAA,CACCTT
4,AAAGTG,GTGAAA,CACTTT


In [33]:
from Bio import Align
aligner = Align.PairwiseAligner()
aligner.mode = "global"

print(aligner)

aligner.open_left_deletion_score = 0.000
aligner.extend_left_deletion_score = 0.000
aligner.open_right_deletion_score = 0.000
aligner.extend_right_deletion_score = 0.000

print(aligner)

def get_score(row):
    return aligner.score(row["gene"], row["seed6mer_rvc"])

for name, df in datasets.items():
    print(f'-------{name}------')
    df["score"] = df.apply(get_score, axis=1)
    display(df[["gene", "noncodingRNA", "score"]].head())


Pairwise sequence aligner with parameters
  wildcard: None
  match_score: 1.000000
  mismatch_score: 0.000000
  open_internal_insertion_score: -1.000000
  extend_internal_insertion_score: -1.000000
  open_left_insertion_score: -1.000000
  extend_left_insertion_score: -1.000000
  open_right_insertion_score: -1.000000
  extend_right_insertion_score: -1.000000
  open_internal_deletion_score: -1.000000
  extend_internal_deletion_score: -1.000000
  open_left_deletion_score: -1.000000
  extend_left_deletion_score: -1.000000
  open_right_deletion_score: -1.000000
  extend_right_deletion_score: -1.000000
  mode: global

Pairwise sequence aligner with parameters
  wildcard: None
  match_score: 1.000000
  mismatch_score: 0.000000
  open_internal_insertion_score: -1.000000
  extend_internal_insertion_score: -1.000000
  open_left_insertion_score: -1.000000
  extend_left_insertion_score: -1.000000
  open_right_insertion_score: -1.000000
  extend_right_insertion_score: -1.000000
  open_internal_dele

,gene,noncodingRNA,score
0,GGAGTCTGGAGTCAAACCCAGAGCAGCTGCAGGCCATGAGGCACAT...,AAAGCAAATGTTGGGTGAACGGC,4.0
1,CAGCTGTGTACAGCGCCATCTCTCTGCCTTCTGTTGCCCCTCACTC...,AATAGCTCAGAATGTCAGTTCTG,5.0
2,GCCTCATGCGCTCACGGGTTCGGGGGGCCGATGCTGCCCAAGCCAA...,AGAATTCTCTTATCCAACATCAACA,4.0
3,AAGAAAATGGTTCAAGGGCATGGGGGTTAGAGAATGTTTCTTTTAC...,GCTAAGGAAGTCCTGTGCTCAG,4.0
4,CCACCTGCAACCCCAATCTCAATCCGGAGGCCAGTAGCACCAGGAT...,TGAAGCGCCTGTGCTCTGCCGAGA,3.0


-------AGO2_eCLIP_Klimentova2022------


,gene,noncodingRNA,score
0,GCTAACAACATCTGCACAGCACCTTACAGTTTGCAAAGAACGTTCA...,TAAGGTGCATCTAGTGCAGATAG,6.0
1,TCTTCCTTTTCCCCACGTTCAGTGTAATCTCACTGAACAGTAATAA...,TAAGGTGCATCTAGTGCAGATAG,5.0
2,ACAGAGCCATATGTACCCTACCCTGCACATTGTTATGCACTTTTCC...,AAAAGTGCTTACAGTGCAGGTAG,6.0
3,CACTCCATCAGCACTAGCACCTCACTCCATCGGCCCCGGCACCCTG...,TAAGGTGCATCTAGTGCAGATAG,5.0
4,GTAAGCACTTTATCTCCAGAACTGAGAGCAAAGTTAACAAATCTCA...,TAAAGTGCTTATAGTGCAGGTAG,6.0
